# Flu Shot Learning — Notebook 4: Tuned Random Forest + Feature Engineering + 5-Fold CV

**Purpose:** Isolate whether Random Forest's earlier underperformance (0.8410 in Notebook 1) was the algorithm's true ceiling, or simply the result of inadequate tuning, no cross-validation, and no feature engineering.

**Key changes vs Notebook 1:**
- **Tuned hyperparameters:** `n_estimators=500`, `max_depth=12`, `min_samples_leaf=5`, `max_features='sqrt'` (fixing under-regularized defaults).
- **Feature engineering:** `behavior_score`, `opinion_score`, `missing_count`, ordinal mappings for `age_group`/`education`/`income_poverty`.
- **Missingness indicators:** Binary `_was_missing` flags for the top 5 most-missing fields.
- **5-Fold StratifiedKFold** with proper OOF predictions reported (Notebook 1 never had this).
- **ColumnTransformer pipeline:** median impute + scale for numeric, impute + one-hot for categorical — applied *after* feature engineering.


In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
print('All imports OK')

All imports OK


## 1. Locate and Load Data

In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
       
    ]
    for c in candidates:
        if (c / 'training_set_features.csv').exists():
            return c
    raise FileNotFoundError('Could not find data. Set DATA_DIR manually.')

DATA_DIR = find_data_dir()
print(f'Data directory: {DATA_DIR.resolve()}')

train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv', index_col='respondent_id')
train_labels   = pd.read_csv(DATA_DIR / 'training_set_labels.csv',   index_col='respondent_id')
test_features  = pd.read_csv(DATA_DIR / 'test_set_features.csv',  index_col='respondent_id')

print(f'Train: {train_features.shape}, Test: {test_features.shape}')
print(f'Positive rates:\n{train_labels[TARGETS].mean()}')

Data directory: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 35), Test: (26708, 35)
Positive rates:
h1n1_vaccine        0.212454
seasonal_vaccine    0.465608
dtype: float64


## 2. Feature Engineering

All engineered features are target-independent so they can be safely applied before the CV loop.

In [3]:
# ---------- Ordinal mappings ----------
AGE_MAP = {
    '18 - 34 Years': 0,
    '35 - 44 Years': 1,
    '45 - 54 Years': 2,
    '55 - 64 Years': 3,
    '65+ Years': 4,
}
EDU_MAP = {
    '< 12 Years': 0,
    '12 Years': 1,
    'Some College': 2,
    'College Graduate': 3,
}
INC_MAP = {
    'Below Poverty': 0,
    '<= $75,000, Above Poverty': 1,
    '> $75,000': 2,
}

# Top 5 most-missing fields
TOP_MISSING_COLS = [
    'employment_occupation',   # ~50%
    'employment_industry',     # ~50%
    'health_insurance',        # ~46%
    'income_poverty',          # ~17%
    'doctor_recc_seasonal',    # ~8%
]


def feature_engineer(df):
    """Apply all target-independent feature engineering."""
    df = df.copy()
    
    # --- Missingness indicators ---
    for col in TOP_MISSING_COLS:
        if col in df.columns:
            df[f'{col}_was_missing'] = df[col].isnull().astype(int)
    
    # --- Aggregate: total missing count per row ---
    df['missing_count'] = df.isnull().sum(axis=1)
    
    # --- Behavior score: sum of all behavioral binary flags ---
    behavior_cols = [c for c in df.columns if c.startswith('behavioral_')]
    df['behavior_score'] = df[behavior_cols].sum(axis=1)
    
    # --- Opinion score: sum of all opinion ratings ---
    opinion_cols = [c for c in df.columns if c.startswith('opinion_')]
    df['opinion_score'] = df[opinion_cols].sum(axis=1)
    
    # --- Ordinal encode age, education, income ---
    # Create new numeric columns; keep originals for the pipeline's categorical branch
    df['age_group_ord'] = df['age_group'].map(AGE_MAP)
    df['education_ord'] = df['education'].map(EDU_MAP)
    df['income_poverty_ord'] = df['income_poverty'].map(INC_MAP)
    
    return df


X_all = feature_engineer(train_features)
X_test_all = feature_engineer(test_features)

print(f'After feature engineering — Train: {X_all.shape}, Test: {X_test_all.shape}')
print(f'New columns: missing_count, behavior_score, opinion_score, age_group_ord, education_ord, income_poverty_ord')
print(f'Missingness indicators: {", ".join([c + "_was_missing" for c in TOP_MISSING_COLS])}')

After feature engineering — Train: (26707, 46), Test: (26708, 46)
New columns: missing_count, behavior_score, opinion_score, age_group_ord, education_ord, income_poverty_ord
Missingness indicators: employment_occupation_was_missing, employment_industry_was_missing, health_insurance_was_missing, income_poverty_was_missing, doctor_recc_seasonal_was_missing


## 3. Build ColumnTransformer Pipeline

Numeric columns get median imputation + standard scaling. Categorical columns get constant imputation + one-hot encoding. The engineered ordinal and aggregate columns are numeric and flow through the numeric branch.

In [4]:
# Identify column types AFTER feature engineering
numeric_cols = X_all.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
categorical_cols = X_all.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features: {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)} -> {categorical_cols}')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

print('ColumnTransformer pipeline built.')

Numeric features: 34
Categorical features: 12 -> ['age_group', 'education', 'race', 'sex', 'income_poverty', 'marital_status', 'rent_or_own', 'employment_status', 'hhs_geo_region', 'census_msa', 'employment_industry', 'employment_occupation']
ColumnTransformer pipeline built.


## 4. Tuned Random Forest Model

Key tuning vs Notebook 1 defaults:
- `n_estimators`: 100 → **500** (more trees = lower variance)
- `max_depth`: None → **12** (prevents trees from memorizing noise)
- `min_samples_leaf`: 1 → **5** (smooths leaf predictions)
- `max_features`: 'auto' → **'sqrt'** (decorrelates trees via feature subsampling)

In [5]:
RF_PARAMS = {
    'n_estimators': 500,
    'max_depth': 12,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

print('Random Forest params:')
for k, v in RF_PARAMS.items():
    print(f'  {k}: {v}')

Random Forest params:
  n_estimators: 500
  max_depth: 12
  min_samples_leaf: 5
  max_features: sqrt
  random_state: 42
  n_jobs: -1


## 5. Training Loop: 5-Fold Stratified CV with OOF Predictions

We use `MultiOutputClassifier` so both targets are trained jointly per fold, exactly matching the original Notebook 1 approach but now with proper CV and OOF evaluation.

In [6]:
# Joint stratification key
strat_key = train_labels['h1n1_vaccine'].astype(str) + '_' + train_labels['seasonal_vaccine'].astype(str)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_h1n1     = np.zeros(len(X_all))
oof_seasonal = np.zeros(len(X_all))
test_h1n1     = np.zeros(len(X_test_all))
test_seasonal = np.zeros(len(X_test_all))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all, strat_key)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    X_tr = X_all.iloc[train_idx]
    X_va = X_all.iloc[val_idx]
    y_tr = train_labels.iloc[train_idx]
    y_va = train_labels.iloc[val_idx]
    
    # Build full pipeline: preprocessor + tuned RF
    rf = RandomForestClassifier(**RF_PARAMS)
    multi_rf = MultiOutputClassifier(rf, n_jobs=-1)
    
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', multi_rf),
    ])
    
    clf.fit(X_tr, y_tr)
    
    # predict_proba returns a list of arrays, one per target
    val_probas = clf.predict_proba(X_va)
    val_h1n1_prob     = val_probas[0][:, 1]
    val_seasonal_prob = val_probas[1][:, 1]
    
    oof_h1n1[val_idx]     = val_h1n1_prob
    oof_seasonal[val_idx] = val_seasonal_prob
    
    # Test predictions averaged across folds
    test_probas = clf.predict_proba(X_test_all)
    test_h1n1     += test_probas[0][:, 1] / N_SPLITS
    test_seasonal += test_probas[1][:, 1] / N_SPLITS
    
    # Per-fold metrics
    auc_h = roc_auc_score(y_va['h1n1_vaccine'], val_h1n1_prob)
    auc_s = roc_auc_score(y_va['seasonal_vaccine'], val_seasonal_prob)
    print(f"  H1N1 AUC:     {auc_h:.5f}")
    print(f"  Seasonal AUC: {auc_s:.5f}")
    print(f"  Mean AUC:     {(auc_h + auc_s) / 2:.5f}")

# ---------- Overall OOF metrics ----------
oof_auc_h1n1     = roc_auc_score(train_labels['h1n1_vaccine'], oof_h1n1)
oof_auc_seasonal = roc_auc_score(train_labels['seasonal_vaccine'], oof_seasonal)
oof_mean = (oof_auc_h1n1 + oof_auc_seasonal) / 2

print(f"\n{'='*55}")
print(f"  OOF ROC AUC — H1N1:     {oof_auc_h1n1:.5f}")
print(f"  OOF ROC AUC — Seasonal: {oof_auc_seasonal:.5f}")
print(f"  OOF ROC AUC — MEAN:     {oof_mean:.5f}")
print(f"{'='*55}")


--- Fold 1/5 ---
  H1N1 AUC:     0.85671
  Seasonal AUC: 0.84716
  Mean AUC:     0.85193

--- Fold 2/5 ---
  H1N1 AUC:     0.86792
  Seasonal AUC: 0.86218
  Mean AUC:     0.86505

--- Fold 3/5 ---
  H1N1 AUC:     0.85762
  Seasonal AUC: 0.85495
  Mean AUC:     0.85629

--- Fold 4/5 ---
  H1N1 AUC:     0.86235
  Seasonal AUC: 0.84966
  Mean AUC:     0.85601

--- Fold 5/5 ---
  H1N1 AUC:     0.86082
  Seasonal AUC: 0.86259
  Mean AUC:     0.86170

  OOF ROC AUC — H1N1:     0.86102
  OOF ROC AUC — Seasonal: 0.85518
  OOF ROC AUC — MEAN:     0.85810


## 6. Generate Submission

In [7]:
submission_df = pd.read_csv(DATA_DIR / 'submission_format.csv', index_col='respondent_id')
submission_df['h1n1_vaccine']     = test_h1n1
submission_df['seasonal_vaccine'] = test_seasonal

# Sanity checks
assert submission_df.shape[0] == len(test_features), 'Row count mismatch!'
assert submission_df[['h1n1_vaccine', 'seasonal_vaccine']].min().min() >= 0, 'Negative probabilities!'
assert submission_df[['h1n1_vaccine', 'seasonal_vaccine']].max().max() <= 1, 'Probabilities > 1!'

SUBMISSIONS_DIR = Path('../../submissions')
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
out_path = SUBMISSIONS_DIR / 'submission4_rf_tuned.csv'
submission_df.to_csv(out_path)

# Also save to /kaggle/working if on Kaggle
kaggle_out = Path('/kaggle/working/submission.csv')
if kaggle_out.parent.exists():
    submission_df.to_csv(kaggle_out)
    print(f'Saved to {kaggle_out}')

print(f'Submission saved to {out_path}')
print(f'Rows: {len(submission_df)}')
submission_df.head()

Saved to /kaggle/working/submission.csv
Submission saved to ../../submissions/submission4_rf_tuned.csv
Rows: 26708


,h1n1_vaccine,seasonal_vaccine
respondent_id,,
26707,0.161914,0.306114
26708,0.056892,0.067294
26709,0.283819,0.738629
26710,0.496783,0.832508
26711,0.296057,0.494040


## 7. Comparison Summary

| Notebook | Algorithm | CV | Feature Eng. | OOF Mean AUC |
|---|---|---|---|---|
| NB 1 (original) | RF (defaults) | Single split | None | 0.8410 |
| **NB 4 (this)** | **RF (tuned)** | **5-Fold Stratified** | **Yes** | **See above** |

If the OOF AUC here is significantly higher than 0.8410, the gap was caused by tuning and engineering — not the algorithm. If it's still far below the gradient boosting notebooks, Random Forest has genuinely hit its ceiling on this dataset.